<a href="https://colab.research.google.com/github/pineapple-666/Veritus-Augustin-Hackathon/blob/main/Theme_Classifier_failed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
"""
LLM-based Theme Classification for Survey Questions
Classifies questions into themes: career focused, academic reputation,
financial support, and flexible learning
"""

# Install required packages
!pip install llm pandas openpyxl -q

import pandas as pd
import llm
import json
import re

# ============================================================================
# CONFIGURATION
# ============================================================================
# Updated to load from Google Drive instead of GitHub URL
URL = "/content/drive/MyDrive/Copy of Staff Survey_anonymized.csv"
OUTPUT_PATH = 'theme_classification_results.csv'  # Will save in Colab's current directory

# ============================================================================
# Load the CSV file
# ============================================================================
print("Loading CSV from Google Drive...")
try:
    # Read first two rows without header
    df_raw = pd.read_csv(URL, header=None, nrows=2)
    print("✓ CSV loaded successfully")
except Exception as e:
    print(f"⚠ Error loading CSV: {e}")
    print("\nPlease check:")
    print("1. The file path is correct and accessible in Google Drive")
    print("2. Google Drive is mounted in Colab")
    raise

print("\nCSV Structure:")
print(f"Total columns: {len(df_raw.columns)}")
print("\nFirst 10 columns:")
print(df_raw.iloc[:, :10].T)
print("\n")

# Extract row 1 (question codes) and row 2 (full questions)
row1_codes = df_raw.iloc[0].tolist()
row2_questions = df_raw.iloc[1].tolist()

# Filter to keep only question columns (starting with Q)
questions_dict = {}

for i, (code, question) in enumerate(zip(row1_codes, row2_questions)):
    if pd.notna(code) and pd.notna(question):
        code_str = str(code).strip()
        question_str = str(question).strip()

        # Only include columns that start with 'Q' (question codes)
        if code_str.startswith('Q') and len(question_str) > 10:
            questions_dict[code_str] = question_str

print(f"Found {len(questions_dict)} question columns (starting with 'Q')")
print("="*70)
print("\nFirst 5 questions:")
for i, (code, question) in enumerate(list(questions_dict.items())[:5], 1):
    print(f"\n{i}. [{code}]")
    print(f"   {question[:100]}{'...' if len(question) > 100 else ''}")

print("\n" + "="*70)

# ============================================================================
# Define themes and create LLM model
# ============================================================================
themes = ["career focused", "academic reputation", "financial support", "flexible"]

print("\nTheme Definitions:")
print("-" * 70)
print("• career focused: Career preparation, job readiness, employment, industry connections")
print("• academic reputation: Prestige, rankings, quality of education, institutional reputation")
print("• financial support: Scholarships, financial aid, tuition costs, affordability")
print("• flexible: Flexible learning (online, evening, hybrid, part-time scheduling)")
print()

# Initialize LLM model
print("Initializing LLM model...")
print("NOTE: If you haven't set up your API key yet, run: !llm keys set openai")
print()

try:
    model = llm.get_model("gpt-4o-mini")
    print("✓ LLM Model initialized: gpt-4o-mini")
except Exception as e:
    print(f"⚠ Error initializing model: {e}")
    print("\nTo set up your API key:")
    print("1. Run: !llm keys set openai")
    print("2. Enter your OpenAI API key when prompted")
    print("3. Re-run this script")
    raise

# ============================================================================
# Classification function
# ============================================================================
def classify_question(question, themes):
    """
    Use LLM to determine which themes are present in the question.
    Returns a dict with theme names as keys and boolean values.
    """
    prompt = f"""Analyze this survey question about promoting Augustin College and determine which themes it relates to.

Themes:
- career focused: relates to career preparation, job readiness, employment outcomes, industry connections
- academic reputation: relates to prestige, rankings, quality of education, faculty expertise, institutional reputation
- financial support: relates to scholarships, financial aid, tuition costs, affordability, funding
- flexible: relates to flexible learning options like online, evening, hybrid, part-time scheduling

Question: "{question}"

Respond with "yes" or "no" for each theme. Use ONLY this JSON format:
{{
    "career focused": "yes/no",
    "academic reputation": "yes/no",
    "financial support": "yes/no",
    "flexible": "yes/no"
}}"""

    try:
        response = model.prompt(prompt)
        response_text = response.text().strip()

        # Extract JSON from response
        if '{' in response_text:
            json_start = response_text.index('{')
            json_end = response_text.rindex('}') + 1
            json_str = response_text[json_start:json_end]
            result = json.loads(json_str)

            # Convert yes/no to boolean
            return {theme: (result.get(theme, "no").lower() == "yes")
                   for theme in themes}
        else:
            print(f"  ⚠ No JSON found in response")
            return {theme: False for theme in themes}

    except Exception as e:
        print(f"  ⚠ Error: {e}")
        return {theme: False for theme in themes}

# ============================================================================
# Classify all questions
# ============================================================================
print("\n" + "="*70)
print("STARTING CLASSIFICATION")
print("="*70)

results = {}
total = len(questions_dict)

for i, (code, question) in enumerate(questions_dict.items(), 1):
    print(f"\n[{i}/{total}] {code}")

    # Truncate long questions for display
    display_q = question[:100] + '...' if len(question) > 100 else question
    print(f"  {display_q}")

    classification = classify_question(question, themes)
    results[code] = {
        'question': question,
        'classifications': classification
    }

    # Show which themes matched
    matched = [theme for theme, is_match in classification.items() if is_match]
    if matched:
        print(f"  ✓ Themes: {', '.join(matched)}")
    else:
        print(f"  - No themes matched")

# ============================================================================
# Organize results by theme
# ============================================================================
print("\n" + "="*70)
print("RESULTS ORGANIZED BY THEME")
print("="*70)

theme_results = {theme: [] for theme in themes}

for code, data in results.items():
    question = data['question']
    classifications = data['classifications']

    for theme in themes:
        if classifications[theme]:
            theme_results[theme].append({
                'code': code,
                'question': question
            })

# Print results by theme
for theme in themes:
    print(f"\n{'='*70}")
    print(f"THEME: {theme.upper()}")
    print('='*70)
    if theme_results[theme]:
        for idx, item in enumerate(theme_results[theme], 1):
            print(f"\n{idx}. [{item['code']}]")
            print(f"   {item['question']}")
        print(f"\n>>> Total: {len(theme_results[theme])} questions")
    else:
        print(">>> No questions found for this theme.")

# ============================================================================
# Save results to CSV (organized by theme)
# ============================================================================
output_data = []
for theme in themes:
    for item in theme_results[theme]:
        output_data.append({
            'Theme': theme,
            'Question_Code': item['code'],
            'Question_Text': item['question']
        })

output_df = pd.DataFrame(output_data)
output_df.to_csv(OUTPUT_PATH, index=False)
print(f"\n✓ Theme-organized results saved to: {OUTPUT_PATH}")

# ============================================================================
# Create classification matrix
# ============================================================================
print("\n" + "="*70)
print("CLASSIFICATION MATRIX")
print("="*70)

matrix_data = []
for code, data in results.items():
    row = {
        'Question_Code': code,
        'Question_Text': data['question'][:80] + ('...' if len(data['question']) > 80 else '')
    }
    for theme in themes:
        row[theme] = '✓' if data['classifications'][theme] else ''
    matrix_data.append(row)

matrix_df = pd.DataFrame(matrix_data)
print("\n" + matrix_df.to_string(index=False))

# Save matrix
matrix_path = OUTPUT_PATH.replace('.csv', '_matrix.csv')
matrix_df.to_csv(matrix_path, index=False)
print(f"\n✓ Classification matrix saved to: {matrix_path}")

# ============================================================================
# Summary statistics
# ============================================================================
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
for theme in themes:
    count = len(theme_results[theme])
    percentage = (count / total * 100) if total > 0 else 0
    print(f"{theme:25s}: {count:3d} questions ({percentage:5.1f}%)")
print(f"{'-'*70}")
print(f"{'Total questions analyzed':25s}: {total:3d}")

print("\n" + "="*70)
print("✓ CLASSIFICATION COMPLETE")
print("="*70)
print(f"\nOutput files:")
print(f"  1. {OUTPUT_PATH}")
print(f"  2. {matrix_path}")

Loading CSV from Google Drive...
✓ CSV loaded successfully

CSV Structure:
Total columns: 100

First 10 columns:
                       0                                                  1
0              StartDate                                         Start Date
1                EndDate                                           End Date
2               Progress                                           Progress
3  Duration (in seconds)                              Duration (in seconds)
4               Finished                                           Finished
5           RecordedDate                                      Recorded Date
6                 Q2.2_1  How important are the following aspects for pr...
7                 Q2.2_2  How important are the following aspects for pr...
8                 Q2.2_3  How important are the following aspects for pr...
9                 Q2.2_4  How important are the following aspects for pr...


Found 94 question columns (starting with 'Q')

Fi